# Intelli-Speech — EXP2_STAGE_A (measurement validity)

This notebook runs Stage A of Experiment 2 (plan v0.2.1). It checks, on **typical Telugu speech only**, whether the label-free articulation and pause measures are valid and robust to the phone channel.

**What this notebook does not do:**
- Compare atypical with typical speakers (that is Stage B, not implemented).
- Diagnose any speaker or condition.
- Apply any synthetic articulation transform.

**Runtime:** GPU (T4) recommended for cell 5 (alignment); everything else runs on CPU.

**Outputs:** `Intelli-Speech-data/analysis_v2/exp2_stage_a/`. Speakers appear only as T1… or S1… in printed output. **Do not commit this notebook with outputs.**

In [14]:
# CELL 1 | LOGIN + SYNC + DEPENDENCIES
from huggingface_hub import notebook_login, whoami
try:
    whoami()
except Exception:
    notebook_login()
import os, subprocess, sys, json, shutil
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "https://github.com/niharika9nx/Intelli-Speech.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
assert Path("scripts/exp2_stage_a.py").is_file(), "Pull the latest code first."
!pip install -q datasets soundfile praat-parselmouth "librosa==0.10.2.post1" "numpy<2.5"
!pip install -q pyworld || echo "pyworld install failed -> detectability will be reported as incomplete"
DATA_ROOT = "/content/drive/MyDrive/Intelli-Speech-data"
HF_CACHE = f"{DATA_ROOT}/hf_cache"
OUT = Path(DATA_ROOT) / "analysis_v2" / "exp2_stage_a"
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
enc = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
print("device:", DEVICE, "| libopus:", "libopus" in enc, "| libopencore_amrnb:", "libopencore_amrnb" in enc)
try:
    import torchaudio; print("torchaudio", torchaudio.__version__, "| MMS_FA available:", hasattr(torchaudio.pipelines, "MMS_FA"))
except Exception as e:
    print("torchaudio unavailable:", e)
RUN = f'python scripts/exp2_stage_a.py --data-root "{DATA_ROOT}" --hf-cache "{HF_CACHE}" --device {DEVICE}'

device: cuda | libopus: True | libopencore_amrnb: False
torchaudio 2.11.0+cu130 | MMS_FA available: True


In [5]:
# CELL 2 | SMOKE TEST (5 clips per phase, separate folder; safe to re-run)
SMOKE = Path(DATA_ROOT) / "analysis_v2" / "exp2_stage_a_smoke"
!{RUN} --output-dir "{SMOKE}" --phase measure --limit 5 --overwrite
!{RUN} --output-dir "{SMOKE}" --phase synthetic --overwrite
!{RUN} --output-dir "{SMOKE}" --phase channel --limit 5 --overwrite
!{RUN} --output-dir "{SMOKE}" --phase align --limit 5 --overwrite

{
 "recordings": 5,
 "errors": 0,
 "gate_pass_share": 0.8,
 "speakers": 2,
 "formant_sufficient_speakers": 0
}
{
 "f1_median_rel_err": 0.06721234685442876,
 "f2_median_rel_err": 0.08711718005824165,
 "pass": true
}
{
 "conditions": {
  "lp3000": "run",
  "lp3400": "run",
  "lp4000": "run",
  "hp300": "run",
  "g711_mulaw_8k": "run",
  "opus_8k": "run",
  "opus_12k": "run",
  "amrnb_12k2": "not_run"
 },
 "ffmpeg_encoders_used": [
  "libopus"
 ]
}
{
 "backend": "torchaudio",
 "utterances": 5,
 "status_counts": {
  "aligned": 3,
  "low_confidence": 2
 },
 "failure_reasons": {},
 "labelled_tokens": 23,
 "seconds": 4.0
}


In [6]:
# CELL 3 | MEASURE typical clips (controls_v2 + reference sample; ~10-20 min CPU)
!{RUN} --phase measure

[measure] 100/1326 (39s)
[measure] 200/1326 (72s)
[measure] 300/1326 (101s)
[measure] 400/1326 (132s)
[measure] 500/1326 (163s)
[measure] 600/1326 (195s)
[measure] 700/1326 (226s)
[measure] 800/1326 (257s)
[measure] 900/1326 (288s)
[measure] 1000/1326 (319s)
[measure] 1100/1326 (349s)
[measure] 1200/1326 (383s)
[measure] 1300/1326 (417s)
{
 "recordings": 1326,
 "errors": 0,
 "gate_pass_share": 0.9728506787330317,
 "speakers": 440,
 "formant_sufficient_speakers": 95
}


In [7]:
# CELL 4 | SYNTHETIC RECOVERY + CHANNEL ROBUSTNESS (~5-15 min CPU)
!{RUN} --phase synthetic
!{RUN} --phase channel

{
 "f1_median_rel_err": 0.06721234685442876,
 "f2_median_rel_err": 0.08711718005824165,
 "pass": true
}
[lp3000] 100/126 (37s)
[lp3400] 100/126 (37s)
[lp4000] 100/126 (37s)
[hp300] 100/126 (39s)
[g711_mulaw_8k] 100/126 (37s)
[opus_8k] 100/126 (75s)
[opus_12k] 100/126 (82s)
{
 "conditions": {
  "lp3000": "run",
  "lp3400": "run",
  "lp4000": "run",
  "hp300": "run",
  "g711_mulaw_8k": "run",
  "opus_8k": "run",
  "opus_12k": "run",
  "amrnb_12k2": "not_run"
 },
 "ffmpeg_encoders_used": [
  "libopus"
 ]
}


In [8]:
# CELL 5 | MMS FORCED ALIGNMENT of verbatim typical transcripts (GPU recommended; downloads ~1.2 GB model once)
!{RUN} --phase align

[align] 50/482 (26s)
[align] 100/482 (45s)
[align] 150/482 (67s)
[align] 200/482 (89s)
[align] 250/482 (109s)
[align] 300/482 (125s)
[align] 350/482 (139s)
[align] 400/482 (161s)
[align] 450/482 (181s)
{
 "backend": "torchaudio",
 "utterances": 482,
 "status_counts": {
  "aligned": 457,
  "excluded": 19,
  "low_confidence": 6
 },
 "failure_reasons": {
  "bracketed_telugu": 19
 },
 "labelled_tokens": 2891,
 "seconds": 191.8
}


In [15]:
# CELL 6 | TRANSFORM DETECTABILITY (needs pyworld; reported as incomplete if missing) (~10-20 min CPU)
!{RUN} --phase detectability

/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
{
 "status": "run",
 "n_clips": 50,
 "measures": {
  "aavs_log": {
   "status": "pass",
   "kind": "warp",
   "medians": [
    0.03777829653795939,
    0.03192883666955303,
    0.02745550788427356,
    0.022609051125339903
   ]
  },
  "vsa_hull_log": {
   "status": "pass",
   "kind": "warp",
   "medians": [
    0.304865572582836,
    0.2667790764205891,
    0.2189527754768415,
    0.17855071263134722
   ]
  },
  "f2_excursion_st_median": {
   "status": "fail",
   "kind": "smooth",
   "medians": [
    2.2871736883169325,
    2.4284404659421552,
    3.0537940693861882,
    3.477197503554458
   ]
  },
  "f2_speed_fit_oct_s": {
   "status": "fail",
   "kind": "smooth

In [10]:
# CELL 7 | EXPORT THE 30-UTTERANCE MANUAL ALIGNMENT CHECK (never overwritten)
!{RUN} --phase manual-sheet
print("Manual-check folder:", OUT / "manual_check")

{
 "utterances": 30,
 "speakers": 28,
 "tokens": 180,
 "category_coverage_utterances": {
  "a": 30,
  "aa": 29,
  "i": 30,
  "ii": 24,
  "u": 30,
  "uu": 23,
  "e": 22,
  "ee": 27,
  "o": 23,
  "oo": 25
 },
 "low_confidence_utterances": 13
}
Manual-check folder: /content/drive/MyDrive/Intelli-Speech-data/analysis_v2/exp2_stage_a/manual_check


### Manual alignment check (done by a native Telugu speaker, outside Colab)

1. **Download.** From Drive, download `analysis_v2/exp2_stage_a/manual_check/`.
2. **Open each utterance in Praat.** For each `utt_id`, open its `.wav` and `.TextGrid` together: select both, then **View & Edit**.
   - The **words** tier shows the transcript words.
   - The **vowels** tier shows the automatic vowel boundaries and their category (a, aa, i, ii, …).
3. **Fill `tokens_sheet.csv`, one row per listed vowel token.**
   - `human_start_s`, `human_end_s`:
     - type `=` if the automatic boundary is correct;
     - otherwise type the corrected time in seconds;
     - leave both empty only if you did not check that row.
   - `vowel_present`: `y` if that stretch really contains the vowel, else `n`.
   - `category_correct`: `y` if the vowel is the labelled category, including short vs long, else `n`.
4. **Fill `utterances_sheet.csv`.** `gross_misalignment`: `y` if any word is misplaced by more than its own length, else `n`.
5. **Upload.** Put the two filled sheets back in the same Drive folder, keeping the same file names.
6. **Run cell 8.**

**Acceptance is pre-registered and will not be changed:**
- median absolute boundary error ≤ 30 ms;
- at least 80 % of boundaries within 50 ms;
- utterance failure rate ≤ 20 %;
- category agreement ≥ 85 %.

In [11]:
# CELL 8 | MANUAL CHECK SUMMARY (run after uploading the filled sheets)
!{RUN} --phase manual-summary --overwrite   # derived output; re-run after uploading the filled sheets

{
 "n_tokens_checked": 0,
 "n_boundaries": 0,
 "n_utterances_checked": 0,
 "median_abs_boundary_ms": null,
 "p90_abs_boundary_ms": null,
 "within_20ms": null,
 "within_50ms": null,
 "utterance_failure_rate": null,
 "category_agreement": null,
 "spearman_conf_vs_error": null,
 "by_group": {},
 "acceptance": "incomplete"
}


In [12]:
# CELL 9 | LEXICAL COMPARABILITY (text only) + ATYPICAL FEASIBILITY (sealed: ICCs and counts only)
!{RUN} --phase lexical
!{RUN} --phase atypical-feasibility

README.md: 100% 7.27k/7.27k [00:00<00:00, 15.5MB/s]
{
 "control_shares": {
  "i": 0.24244591453602718,
  "a": 0.520561416055784,
  "u": 0.2369926694081888
 },
 "atypical_shares": {
  "i": 0.25655808590371865,
  "a": 0.486739694436437,
  "u": 0.25670221965984436
 },
 "max_share_diff": 0.03382172161934699,
 "js_distance": 0.028939637771406023,
 "flag": false,
 "control_counts": {
  "a": 3830,
  "aa": 1993,
  "i": 2394,
  "ii": 318,
  "u": 2232,
  "uu": 419,
  "e": 505,
  "ee": 547,
  "o": 381,
  "oo": 424
 },
 "atypical_counts": {
  "a": 2233,
  "aa": 1144,
  "i": 1582,
  "ii": 198,
  "u": 1654,
  "uu": 127,
  "e": 212,
  "ee": 220,
  "o": 234,
  "oo": 418
 },
 "note": "Text only; atypical transcripts are intended meaning."
}
Traceback (most recent call last):
  File "/content/Intelli-Speech/scripts/exp2_stage_a.py", line 452, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/Intelli-Speech/scripts/exp2_stage_a.py", line 443, in main
    result = PHASE

In [ ]:
# CELL 10 | STAGE A REPORT
!{RUN} --phase report --overwrite
import pandas as pd
rep = json.loads((OUT / "stage_a_report.json").read_text())
rows = []
for m, e in rep["measures"].items():
    rows.append({"measure": m, "overall": e["overall"], **{t: v.get("status") for t, v in e["tests"].items()}})
print(pd.DataFrame(rows).fillna("—").to_string(index=False))
print(); print("manual alignment:", (rep.get("manual_alignment") or {}).get("acceptance"))
print("lexical flag:", (rep.get("lexical_flag") or {}).get("flag"))

Save the notebook and tell Claude. Do not commit it with outputs.